# Governance Dashboard — QuickSight Analysis for Inference & Drift Monitoring

This notebook programmatically creates a complete QuickSight dashboard — no manual UI steps.

**What it creates (all via API):**
- Athena data source in QuickSight
- Three datasets: `inference_responses`, `monitoring_responses`, and joined feature drift dataset
- Analysis with 16 visuals across 3 sheets (Definition mode)
- Published dashboard with comprehensive monitoring

**Visuals — Sheet 1 (Inference Monitoring):**
1. Prediction Volume Over Time
2. Fraud Probability Distribution
3. Prediction Accuracy Breakdown
4. Risk Tier Distribution
5. Inference Latency Trend
6. Total Inferences KPI

**Visuals — Sheet 2 (Drift Trend Analysis):**
7. Data Drift Share Over Time (line chart)
8. Drifted Columns Count Trend (bar chart)
9. ROC-AUC Degradation Trend (line chart with baseline)
10. Model Performance Metrics Over Time (multi-line: accuracy, precision, recall, F1)
11. Drift Alerts Timeline (bar chart of alert_sent events)
12. Drift Detection KPI (latest drift share)

**Visuals — Sheet 3 (Feature Drift Analysis by Model Version):**
13. Drift by Model Version Over Time (multi-line showing drift evolution per version)
14. Model Version Performance Summary (table with drift, ROC-AUC, run counts)
15. Inference Volume vs Drift Correlation (combo chart showing relationship)
16. Drift Intensity Heatmap (pivot table: time × version)

**Prerequisites:**
- QuickSight Enterprise subscription
- Data in `inference_responses` and `monitoring_responses` Athena tables
- IAM permissions for QuickSight, Athena, S3

**Note on Feature-Level Analysis:**
The `per_feature_drift_scores` column is stored as JSON. For granular per-feature drift visualization, 
see `3_governance_dashboard_feature_drift.md` for options to create an Athena view or separate table.

## 1. Setup & Configuration

In [1]:
import sys, os, json, time, boto3
from pathlib import Path
from botocore.exceptions import ClientError
from dotenv import load_dotenv

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / '.env')

from src.config.config import (
    ATHENA_DATABASE, AWS_DEFAULT_REGION, ATHENA_OUTPUT_S3, DATA_S3_BUCKET,
    ATHENA_INFERENCE_TABLE, ATHENA_MONITORING_RESPONSES_TABLE,
    ATHENA_GROUND_TRUTH_UPDATES_TABLE,
    QUICKSIGHT_DATASOURCE_ID, QUICKSIGHT_DATASOURCE_NAME,
    QUICKSIGHT_INFERENCE_DATASET_ID, QUICKSIGHT_INFERENCE_DATASET_NAME,
    QUICKSIGHT_DRIFT_DATASET_ID, QUICKSIGHT_DRIFT_DATASET_NAME,
    QUICKSIGHT_FEATURE_DRIFT_DATASET_ID, QUICKSIGHT_FEATURE_DRIFT_DATASET_NAME,
    QUICKSIGHT_ANALYSIS_ID, QUICKSIGHT_ANALYSIS_NAME,
    QUICKSIGHT_DASHBOARD_ID, QUICKSIGHT_DASHBOARD_NAME,
    QUICKSIGHT_SERVICE_ROLE_NAME,
)

sts = boto3.client('sts', region_name=AWS_DEFAULT_REGION)
quicksight = boto3.client('quicksight', region_name=AWS_DEFAULT_REGION)
athena = boto3.client('athena', region_name=AWS_DEFAULT_REGION)

ACCOUNT_ID = sts.get_caller_identity()['Account']
REGION = AWS_DEFAULT_REGION

# Use config values instead of hardcoded
DATASOURCE_ID = QUICKSIGHT_DATASOURCE_ID
DATASET_ID = QUICKSIGHT_INFERENCE_DATASET_ID
DRIFT_DATASET_ID = QUICKSIGHT_DRIFT_DATASET_ID
ANALYSIS_ID = QUICKSIGHT_ANALYSIS_ID
DASHBOARD_ID = QUICKSIGHT_DASHBOARD_ID

print(f'Account:  {ACCOUNT_ID}')
print(f'Region:   {REGION}')
print(f'Database: {ATHENA_DATABASE}')

Account:  <ACCOUNT_ID>
Region:   us-east-1
Database: fraud_detection


## 2. Verify QuickSight Subscription

In [2]:
try:
    qs = quicksight.describe_account_settings(AwsAccountId=ACCOUNT_ID)
    edition = qs['AccountSettings'].get('Edition', 'Unknown')
    print(f'\u2713 QuickSight active (Edition: {edition})')
    if edition == 'STANDARD':
        print('  \u26a0 Definition API requires Enterprise edition')
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('\u2717 QuickSight not subscribed: https://quicksight.aws.amazon.com/')
    else: raise

✓ QuickSight active (Edition: ENTERPRISE)


## 3. Verify Inference Data in Athena

In [3]:
def run_athena_query(sql, database=ATHENA_DATABASE):
    r = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={'OutputLocation': ATHENA_OUTPUT_S3},
    )
    eid = r['QueryExecutionId']
    while True:
        s = athena.get_query_execution(QueryExecutionId=eid)
        state = s['QueryExecution']['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'): break
        time.sleep(1)
    if state != 'SUCCEEDED':
        reason = s['QueryExecution']['Status'].get('StateChangeReason', '')
        raise Exception(f'Query {state}: {reason}')
    return athena.get_query_results(QueryExecutionId=eid)

print(f'Checking {ATHENA_INFERENCE_TABLE} table...')
r = run_athena_query(f'SELECT COUNT(*) FROM {ATHENA_INFERENCE_TABLE}')
print(f'  Total records: {r["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"]}')
r = run_athena_query(f'SELECT COUNT(*) FROM {ATHENA_INFERENCE_TABLE} WHERE ground_truth IS NOT NULL')
print(f'  With ground truth: {r["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"]}')
r = run_athena_query(f'SELECT MIN(request_timestamp), MAX(request_timestamp) FROM {ATHENA_INFERENCE_TABLE}')
print(f'  Range: {r["ResultSet"]["Rows"][1]["Data"][0].get("VarCharValue","N/A")} \u2192 {r["ResultSet"]["Rows"][1]["Data"][1].get("VarCharValue","N/A")}')
print('\u2713 Data available')

print(f'\nChecking {ATHENA_MONITORING_RESPONSES_TABLE} table...')
try:
    r = run_athena_query(f'SELECT COUNT(*) FROM {ATHENA_MONITORING_RESPONSES_TABLE}')
    drift_count = r['ResultSet']['Rows'][1]['Data'][0]['VarCharValue']
    print(f'  Total monitoring runs: {drift_count}')
    r = run_athena_query(f'SELECT MIN(monitoring_timestamp), MAX(monitoring_timestamp) FROM {ATHENA_MONITORING_RESPONSES_TABLE}')
    print(f'  Range: {r["ResultSet"]["Rows"][1]["Data"][0].get("VarCharValue","N/A")} \u2192 {r["ResultSet"]["Rows"][1]["Data"][1].get("VarCharValue","N/A")}')
    print('\u2713 Drift data available')
    HAS_DRIFT_DATA = int(drift_count) > 0
except Exception as e:
    print(f'  \u26a0 monitoring_responses not available: {e}')
    print('  Drift trend visuals will be empty until monitoring runs complete.')
    HAS_DRIFT_DATA = False

Checking inference_responses table...
  Total records: 3908
  With ground truth: 3901
  Range: 2026-02-21 15:37:18.814407 → 2026-03-24 14:32:41.019490
✓ Data available

Checking monitoring_responses table...
  Total monitoring runs: 6
  Range: 2026-03-21 09:51:26.000000 → 2026-03-24 07:37:08.000000
✓ Drift data available


## 4. Create Athena Data Source in QuickSight

In [4]:
def get_quicksight_principal():
    try:
        users = quicksight.list_users(AwsAccountId=ACCOUNT_ID, Namespace='default')
        if users['UserList']:
            return users['UserList'][0]['Arn']
    except Exception:
        pass
    return f'arn:aws:quicksight:{REGION}:{ACCOUNT_ID}:user/default/Admin/*'

QS_PRINCIPAL = get_quicksight_principal()
print(f'QuickSight principal: {QS_PRINCIPAL}')

DS_ACTIONS = [
    'quicksight:DescribeDataSource', 'quicksight:DescribeDataSourcePermissions',
    'quicksight:PassDataSource', 'quicksight:UpdateDataSource',
    'quicksight:DeleteDataSource', 'quicksight:UpdateDataSourcePermissions',
]
ds_common = dict(
    AwsAccountId=ACCOUNT_ID, DataSourceId=DATASOURCE_ID,
    Name=QUICKSIGHT_DATASOURCE_NAME,
    DataSourceParameters={'AthenaParameters': {'WorkGroup': 'primary'}},
)
try:
    quicksight.describe_data_source(AwsAccountId=ACCOUNT_ID, DataSourceId=DATASOURCE_ID)
    print('Updating existing data source...')
    resp = quicksight.update_data_source(**ds_common)
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new data source...')
        resp = quicksight.create_data_source(
            **ds_common, Type='ATHENA',
            Permissions=[{'Principal': QS_PRINCIPAL, 'Actions': DS_ACTIONS}],
        )
    else: raise
DATASOURCE_ARN = resp['Arn']
print(f'\u2713 Data source: {DATASOURCE_ARN}')

QuickSight principal: arn:aws:quicksight:us-east-1:<ACCOUNT_ID>:user/default/Admin/skoppar-Isengard
Updating existing data source...
✓ Data source: arn:aws:quicksight:us-east-1:<ACCOUNT_ID>:datasource/fraud-governance-athena-datasource


## 5. Fix Lake Formation Permissions (if needed)

If QuickSight shows **"Lake Formation permissions missing"** when opening the analysis,
uncomment and run the commands below. This grants `IAM_ALLOWED_PRINCIPALS` access so any
IAM role with Glue/Athena permissions (including QuickSight's service role) can query the tables.

In [5]:
# ============================================================================
# LAKE FORMATION & S3 PERMISSIONS FIX
# ============================================================================
# Run this cell if QuickSight shows "Lake Formation permissions missing"
# or "s3:GetObject permission denied" errors.
#
# To grant permissions, call the functions at the bottom of this cell.

import subprocess
import json
import boto3

def grant_lakeformation_permissions(database, tables=None, grant_database=True):
    """
    Grant Lake Formation permissions to IAM_ALLOWED_PRINCIPALS.
    
    Args:
        database: Database name (from ATHENA_DATABASE config)
        tables: List of table names to grant permissions on. If None, only grants database-level.
        grant_database: If True, grants ALL on database first
    
    Returns:
        dict: Results with success/failure status for each resource
    """
    results = {'database': None, 'tables': {}}
    
    def run_grant(resource_json, permissions, resource_name):
        """Execute lakeformation grant-permissions command"""
        cmd = [
            'aws', 'lakeformation', 'grant-permissions',
            '--principal', '{"DataLakePrincipalIdentifier":"IAM_ALLOWED_PRINCIPALS"}',
            '--resource', resource_json,
            '--permissions'
        ] + permissions + ['--region', REGION]
        
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            perms_str = ', '.join(permissions)
            print(f"  ✓ {resource_name}")
            print(f"    Granted: {perms_str}")
            return True
        except subprocess.CalledProcessError as e:
            error_msg = e.stderr.strip()
            # Ignore "already exists" errors
            if 'AlreadyExistsException' in error_msg or 'already granted' in error_msg.lower():
                perms_str = ', '.join(permissions)
                print(f"  ℹ {resource_name}")
                print(f"    Already granted: {perms_str}")
                return True
            else:
                print(f"  ✗ {resource_name}")
                print(f"    Error: {error_msg}")
                return False
    
    print("="*80)
    print(f"LAKE FORMATION PERMISSIONS - Database: {database}")
    print("="*80)
    
    # 1. Grant database-level permissions
    if grant_database:
        resource = json.dumps({"Database": {"Name": database}})
        results['database'] = run_grant(resource, ["ALL"], f"Database: {database}")
        print()
    
    # 2. Grant table-level permissions
    if tables:
        table_permissions = ["SELECT", "DESCRIBE", "ALTER", "DROP", "INSERT", "DELETE"]
        print(f"Granting table permissions ({len(tables)} tables):")
        for i, table in enumerate(tables, 1):
            print(f"\n[{i}/{len(tables)}] Table: {database}.{table}")
            resource = json.dumps({"Table": {"DatabaseName": database, "Name": table}})
            results['tables'][table] = run_grant(
                resource, 
                table_permissions, 
                f"{table}"
            )
    
    return results

def grant_s3_permissions(role_name, bucket_name):
    """
    Grant S3 permissions to QuickSight service role using boto3.
    
    Args:
        role_name: IAM role name (from QUICKSIGHT_SERVICE_ROLE_NAME config)
        bucket_name: S3 bucket name (from DATA_S3_BUCKET config)
    
    Returns:
        dict: Results with success/failure for each policy
    """
    iam = boto3.client('iam', region_name=REGION)
    results = {}
    
    print("\n" + "="*80)
    print(f"S3 PERMISSIONS - Role: {role_name}")
    print("="*80)
    
    # 1. Data lake access policy
    data_lake_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:ListBucket", "s3:GetBucketLocation"],
            "Resource": [
                f"arn:aws:s3:::{bucket_name}",
                f"arn:aws:s3:::{bucket_name}/*"
            ]
        }]
    }
    
    try:
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName='QuickSightS3DataLakeAccess',
            PolicyDocument=json.dumps(data_lake_policy)
        )
        print(f"  ✓ Policy: QuickSightS3DataLakeAccess")
        print(f"    Bucket: {bucket_name}")
        print(f"    Actions: s3:GetObject, s3:ListBucket, s3:GetBucketLocation")
        results['data_lake'] = True
    except Exception as e:
        print(f"  ✗ Policy: QuickSightS3DataLakeAccess")
        print(f"    Error: {e}")
        results['data_lake'] = False
    
    # 2. Athena query results access policy
    athena_results_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket", "s3:GetBucketLocation"],
            "Resource": [
                "arn:aws:s3:::aws-athena-query-results-*",
                "arn:aws:s3:::aws-athena-query-results-*/*"
            ]
        }]
    }
    
    try:
        iam.put_role_policy(
            RoleName=role_name,
            PolicyName='QuickSightAthenaResultsAccess',
            PolicyDocument=json.dumps(athena_results_policy)
        )
        print(f"\n  ✓ Policy: QuickSightAthenaResultsAccess")
        print(f"    Bucket: aws-athena-query-results-* (all regions)")
        print(f"    Actions: s3:GetObject, s3:PutObject, s3:ListBucket, s3:GetBucketLocation")
        results['athena_results'] = True
    except Exception as e:
        print(f"\n  ✗ Policy: QuickSightAthenaResultsAccess")
        print(f"    Error: {e}")
        results['athena_results'] = False
    
    return results

# ============================================================================
# To grant permissions, uncomment and run the code below:
# ============================================================================
#
TABLES_TO_GRANT = [
    ATHENA_INFERENCE_TABLE,
    ATHENA_MONITORING_RESPONSES_TABLE,
    ATHENA_GROUND_TRUTH_UPDATES_TABLE,
]

lf_results = grant_lakeformation_permissions(
    database=ATHENA_DATABASE,
    tables=TABLES_TO_GRANT,
    grant_database=True
)

s3_results = grant_s3_permissions(
    role_name=QUICKSIGHT_SERVICE_ROLE_NAME,
    bucket_name=DATA_S3_BUCKET
)

print("\n" + "="*80)
print("✅ PERMISSIONS SETUP COMPLETE")
print("="*80)

print("Permissions functions defined. Uncomment code above to grant permissions.")

LAKE FORMATION PERMISSIONS - Database: fraud_detection
  ✓ Database: fraud_detection
    Granted: ALL

Granting table permissions (3 tables):

[1/3] Table: fraud_detection.inference_responses
  ✓ inference_responses
    Granted: SELECT, DESCRIBE, ALTER, DROP, INSERT, DELETE

[2/3] Table: fraud_detection.monitoring_responses
  ✓ monitoring_responses
    Granted: SELECT, DESCRIBE, ALTER, DROP, INSERT, DELETE

[3/3] Table: fraud_detection.ground_truth_updates
  ✓ ground_truth_updates
    Granted: SELECT, DESCRIBE, ALTER, DROP, INSERT, DELETE

S3 PERMISSIONS - Role: aws-quicksight-service-role-v0
  ✓ Policy: QuickSightS3DataLakeAccess
    Bucket: fraud-detection-data-lake-skoppar-<ACCOUNT_ID>
    Actions: s3:GetObject, s3:ListBucket, s3:GetBucketLocation

  ✓ Policy: QuickSightAthenaResultsAccess
    Bucket: aws-athena-query-results-* (all regions)
    Actions: s3:GetObject, s3:PutObject, s3:ListBucket, s3:GetBucketLocation

✅ PERMISSIONS SETUP COMPLETE
Permissions functions defined. Uncom

## 6. Create Datasets

Two datasets: `inference_responses` for prediction data, `monitoring_responses` for drift metrics.
Uses `RelationalTable` so QuickSight auto-discovers columns from Athena.
Calculated fields (`prediction_accuracy`, `risk_tier`) are added via `LogicalTableMap`.

In [6]:
# RelationalTable — no column list needed, QuickSight auto-discovers from Athena
physical_table_map = {
    'inference-responses': {
        'RelationalTable': {
            'DataSourceArn': DATASOURCE_ARN,
            'Catalog': 'AwsDataCatalog',
            'Schema': ATHENA_DATABASE,
            'Name': ATHENA_INFERENCE_TABLE,
            'InputColumns': [
                {'Name': 'inference_id', 'Type': 'STRING'},
                {'Name': 'request_timestamp', 'Type': 'DATETIME'},
                {'Name': 'endpoint_name', 'Type': 'STRING'},
                {'Name': 'model_version', 'Type': 'STRING'},
                {'Name': 'mlflow_run_id', 'Type': 'STRING'},
                {'Name': 'input_features', 'Type': 'STRING'},
                {'Name': 'prediction', 'Type': 'INTEGER'},
                {'Name': 'probability_fraud', 'Type': 'DECIMAL'},
                {'Name': 'probability_non_fraud', 'Type': 'DECIMAL'},
                {'Name': 'confidence_score', 'Type': 'DECIMAL'},
                {'Name': 'ground_truth', 'Type': 'INTEGER'},
                {'Name': 'ground_truth_timestamp', 'Type': 'DATETIME'},
                {'Name': 'ground_truth_source', 'Type': 'STRING'},
                {'Name': 'days_to_ground_truth', 'Type': 'DECIMAL'},
                {'Name': 'inference_latency_ms', 'Type': 'DECIMAL'},
                {'Name': 'model_load_time_ms', 'Type': 'DECIMAL'},
                {'Name': 'preprocessing_time_ms', 'Type': 'DECIMAL'},
                {'Name': 'transaction_id', 'Type': 'STRING'},
                {'Name': 'transaction_amount', 'Type': 'DECIMAL'},
                {'Name': 'customer_id', 'Type': 'STRING'},
                {'Name': 'is_high_confidence', 'Type': 'BIT'},
                {'Name': 'is_low_confidence', 'Type': 'BIT'},
                {'Name': 'prediction_bucket', 'Type': 'STRING'},
                {'Name': 'request_id', 'Type': 'STRING'},
                {'Name': 'response_time', 'Type': 'DATETIME'},
                {'Name': 'error_message', 'Type': 'STRING'},
                {'Name': 'inference_mode', 'Type': 'STRING'},
            ],
        }
    }
}

# Calculated fields via LogicalTableMap
logical_table_map = {
    'inference-responses-logical': {
        'Alias': 'Inference Responses',
        'Source': {'PhysicalTableId': 'inference-responses'},
        'DataTransforms': [
            {
                'CreateColumnsOperation': {
                    'Columns': [
                        {
                            'ColumnName': 'prediction_accuracy',
                            'ColumnId': 'prediction-accuracy',
                            'Expression': (
                                "ifelse("
                                "isNull({ground_truth}), 'Pending', "
                                "ifelse({prediction} = {ground_truth}, 'Correct', 'Incorrect'))"
                            ),
                        },
                        {
                            'ColumnName': 'risk_tier',
                            'ColumnId': 'risk-tier',
                            'Expression': (
                                "ifelse("
                                "{probability_fraud} > 0.8, 'High Risk', "
                                "ifelse({probability_fraud} > 0.5, 'Medium Risk', "
                                "ifelse({probability_fraud} > 0.2, 'Low Risk', 'Minimal Risk')))"
                            ),
                        },
                    ]
                }
            }
        ],
    }
}

DSET_ACTIONS = [
    'quicksight:DescribeDataSet', 'quicksight:DescribeDataSetPermissions',
    'quicksight:PassDataSet', 'quicksight:DescribeIngestion',
    'quicksight:ListIngestions', 'quicksight:UpdateDataSet',
    'quicksight:DeleteDataSet', 'quicksight:CreateIngestion',
    'quicksight:CancelIngestion', 'quicksight:UpdateDataSetPermissions',
]
dset_common = dict(
    AwsAccountId=ACCOUNT_ID, DataSetId=DATASET_ID,
    Name=QUICKSIGHT_INFERENCE_DATASET_NAME,
    PhysicalTableMap=physical_table_map,
    LogicalTableMap=logical_table_map,
    ImportMode='DIRECT_QUERY',
)
try:
    quicksight.describe_data_set(AwsAccountId=ACCOUNT_ID, DataSetId=DATASET_ID)
    print('Updating existing dataset...')
    resp = quicksight.update_data_set(**dset_common)
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new dataset...')
        resp = quicksight.create_data_set(
            **dset_common,
            Permissions=[{'Principal': QS_PRINCIPAL, 'Actions': DSET_ACTIONS}],
        )
    else: raise
DATASET_ARN = resp['Arn']
print(f'\u2713 Inference dataset: {DATASET_ARN}')

Updating existing dataset...
✓ Inference dataset: arn:aws:quicksight:us-east-1:<ACCOUNT_ID>:dataset/fraud-governance-inference-dataset


### 6b. Create Drift Monitoring Dataset (monitoring_responses)

In [7]:
# monitoring_responses dataset — drift metrics from Evidently runs
drift_physical_table_map = {
    'monitoring-responses': {
        'RelationalTable': {
            'DataSourceArn': DATASOURCE_ARN,
            'Catalog': 'AwsDataCatalog',
            'Schema': ATHENA_DATABASE,
            'Name': ATHENA_MONITORING_RESPONSES_TABLE,
            'InputColumns': [
                {'Name': 'monitoring_run_id', 'Type': 'STRING'},
                {'Name': 'monitoring_timestamp', 'Type': 'DATETIME'},
                {'Name': 'endpoint_name', 'Type': 'STRING'},
                {'Name': 'model_version', 'Type': 'STRING'},
                {'Name': 'data_drift_detected', 'Type': 'BIT'},
                {'Name': 'drifted_columns_count', 'Type': 'INTEGER'},
                {'Name': 'drifted_columns_share', 'Type': 'DECIMAL'},
                {'Name': 'features_analyzed', 'Type': 'INTEGER'},
                {'Name': 'data_sample_size', 'Type': 'INTEGER'},
                {'Name': 'model_drift_detected', 'Type': 'BIT'},
                {'Name': 'baseline_roc_auc', 'Type': 'DECIMAL'},
                {'Name': 'current_roc_auc', 'Type': 'DECIMAL'},
                {'Name': 'roc_auc_degradation', 'Type': 'DECIMAL'},
                {'Name': 'roc_auc_degradation_pct', 'Type': 'DECIMAL'},
                {'Name': 'accuracy', 'Type': 'DECIMAL'},
                {'Name': 'precision', 'Type': 'DECIMAL'},
                {'Name': 'recall', 'Type': 'DECIMAL'},
                {'Name': 'f1_score', 'Type': 'DECIMAL'},
                {'Name': 'model_sample_size', 'Type': 'INTEGER'},
                {'Name': 'per_feature_drift_scores', 'Type': 'STRING'},
                {'Name': 'evidently_report_s3_path', 'Type': 'STRING'},
                {'Name': 'mlflow_run_id', 'Type': 'STRING'},
                {'Name': 'alert_sent', 'Type': 'BIT'},
                {'Name': 'detection_engine', 'Type': 'STRING'},
                {'Name': 'created_at', 'Type': 'DATETIME'},
            ],
        }
    }
}

drift_logical_table_map = {
    'monitoring-responses-logical': {
        'Alias': 'Monitoring Responses',
        'Source': {'PhysicalTableId': 'monitoring-responses'},
        'DataTransforms': [
            {
                'CreateColumnsOperation': {
                    'Columns': [
                        {
                            'ColumnName': 'drift_severity',
                            'ColumnId': 'drift-severity',
                            'Expression': (
                                "ifelse("
                                "{drifted_columns_share} > 0.3, 'HIGH', "
                                "ifelse({drifted_columns_share} > 0.15, 'MEDIUM', 'LOW'))"
                            ),
                        },
                        {
                            'ColumnName': 'performance_status',
                            'ColumnId': 'performance-status',
                            'Expression': (
                                "ifelse("
                                "{current_roc_auc} >= 0.95, 'GOOD', "
                                "ifelse({current_roc_auc} >= 0.90, 'WARNING', 'CRITICAL'))"
                            ),
                        },
                    ]
                }
            }
        ],
    }
}

drift_dset_common = dict(
    AwsAccountId=ACCOUNT_ID, DataSetId=DRIFT_DATASET_ID,
    Name=QUICKSIGHT_DRIFT_DATASET_NAME,
    PhysicalTableMap=drift_physical_table_map,
    LogicalTableMap=drift_logical_table_map,
    ImportMode='DIRECT_QUERY',
)
try:
    quicksight.describe_data_set(AwsAccountId=ACCOUNT_ID, DataSetId=DRIFT_DATASET_ID)
    print('Updating existing drift dataset...')
    resp = quicksight.update_data_set(**drift_dset_common)
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new drift dataset...')
        resp = quicksight.create_data_set(
            **drift_dset_common,
            Permissions=[{'Principal': QS_PRINCIPAL, 'Actions': DSET_ACTIONS}],
        )
    else: raise
DRIFT_DATASET_ARN = resp['Arn']
print(f'\u2713 Drift dataset: {DRIFT_DATASET_ARN}')

Updating existing drift dataset...
✓ Drift dataset: arn:aws:quicksight:us-east-1:<ACCOUNT_ID>:dataset/fraud-governance-drift-dataset


### 6c. Create Feature Drift Datasets

This section creates two datasets for feature-level drift analysis:

**Dataset 1: Feature Drift by Model Version (Joined)**
- Joins `inference_responses` with `monitoring_responses` by `model_version`
- Shows aggregate drift trends across model versions
- Enables correlation analysis between inference volume and drift

**Dataset 2: Feature-Level Drift Detail (Per-Feature)**
- First creates an Athena view (`feature_drift_detail`) that unpacks the JSON `per_feature_drift_scores` column
- Transforms each monitoring run into multiple rows (one per feature)
- Enables feature-level trendline visualization showing which specific features are drifting over time


In [8]:
# Feature drift dataset — joins monitoring with inference by model_version
# Uses Custom SQL to aggregate inference metrics alongside drift metrics

custom_sql = f'''
SELECT 
    m.monitoring_run_id,
    m.monitoring_timestamp,
    m.model_version,
    m.drifted_columns_count,
    m.drifted_columns_share,
    m.features_analyzed,
    m.baseline_roc_auc,
    m.current_roc_auc,
    m.data_drift_detected,
    m.accuracy,
    m.precision,
    m.recall,
    m.f1_score,
    COUNT(DISTINCT i.inference_id) as inference_count,
    AVG(i.probability_fraud) as avg_fraud_prob,
    COUNT(CASE WHEN i.ground_truth IS NOT NULL THEN 1 END) as gt_count
FROM {ATHENA_DATABASE}.{ATHENA_MONITORING_RESPONSES_TABLE} m
LEFT JOIN {ATHENA_DATABASE}.{ATHENA_INFERENCE_TABLE} i
    ON i.model_version = m.model_version
    AND i.request_timestamp >= m.monitoring_timestamp - INTERVAL '1' DAY
    AND i.request_timestamp < m.monitoring_timestamp
GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13
ORDER BY m.monitoring_timestamp DESC
'''

feature_drift_physical_table_map = {
    'feature-drift-joined': {
        'CustomSql': {
            'DataSourceArn': DATASOURCE_ARN,
            'Name': 'FeatureDriftJoin',
            'SqlQuery': custom_sql,
            'Columns': [
                {'Name': 'monitoring_run_id', 'Type': 'STRING'},
                {'Name': 'monitoring_timestamp', 'Type': 'DATETIME'},
                {'Name': 'model_version', 'Type': 'STRING'},
                {'Name': 'drifted_columns_count', 'Type': 'INTEGER'},
                {'Name': 'drifted_columns_share', 'Type': 'DECIMAL'},
                {'Name': 'features_analyzed', 'Type': 'INTEGER'},
                {'Name': 'baseline_roc_auc', 'Type': 'DECIMAL'},
                {'Name': 'current_roc_auc', 'Type': 'DECIMAL'},
                {'Name': 'data_drift_detected', 'Type': 'BIT'},
                {'Name': 'accuracy', 'Type': 'DECIMAL'},
                {'Name': 'precision', 'Type': 'DECIMAL'},
                {'Name': 'recall', 'Type': 'DECIMAL'},
                {'Name': 'f1_score', 'Type': 'DECIMAL'},
                {'Name': 'inference_count', 'Type': 'INTEGER'},
                {'Name': 'avg_fraud_prob', 'Type': 'DECIMAL'},
                {'Name': 'gt_count', 'Type': 'INTEGER'},
            ],
        }
    }
}

feature_drift_logical_table_map = {
    'feature-drift-logical': {
        'Alias': 'Feature Drift Analysis',
        'Source': {'PhysicalTableId': 'feature-drift-joined'},
    }
}

feature_drift_dset_common = dict(
    AwsAccountId=ACCOUNT_ID, DataSetId=QUICKSIGHT_FEATURE_DRIFT_DATASET_ID,
    Name=QUICKSIGHT_FEATURE_DRIFT_DATASET_NAME,
    PhysicalTableMap=feature_drift_physical_table_map,
    LogicalTableMap=feature_drift_logical_table_map,
    ImportMode='DIRECT_QUERY',
)

try:
    quicksight.describe_data_set(AwsAccountId=ACCOUNT_ID, DataSetId=QUICKSIGHT_FEATURE_DRIFT_DATASET_ID)
    print('Updating existing feature drift dataset...')
    resp = quicksight.update_data_set(**feature_drift_dset_common)
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new feature drift dataset...')
        resp = quicksight.create_data_set(
            **feature_drift_dset_common,
            Permissions=[{'Principal': QS_PRINCIPAL, 'Actions': DSET_ACTIONS}],
        )
    else: raise

FEATURE_DRIFT_DATASET_ARN = resp['Arn']
print(f'✓ Feature drift dataset: {FEATURE_DRIFT_DATASET_ARN}')

Updating existing feature drift dataset...
✓ Feature drift dataset: arn:aws:quicksight:us-east-1:<ACCOUNT_ID>:dataset/fraud-governance-feature-drift-dataset


In [9]:
# Create Athena View for Feature-Level Drift Analysis
# This view unpacks the JSON per_feature_drift_scores into individual rows

import time

print("Creating feature_drift_detail view...")

create_view_sql = f"""
CREATE OR REPLACE VIEW {ATHENA_DATABASE}.feature_drift_detail AS
SELECT
    monitoring_run_id,
    monitoring_timestamp,
    model_version,
    endpoint_name,
    data_drift_detected,
    drifted_columns_count,
    drifted_columns_share,
    baseline_roc_auc,
    current_roc_auc,
    feature_name,                    -- Unpacked from JSON
    drift_score,                     -- Unpacked from JSON
    CASE
        WHEN drift_score > 0.25 THEN 'Significant'
        WHEN drift_score > 0.1 THEN 'Moderate'
        ELSE 'Low'
    END as drift_severity,           -- Computed severity
    CASE WHEN drift_score > 0.1 THEN true ELSE false END as drift_detected
FROM {ATHENA_DATABASE}.monitoring_responses
CROSS JOIN UNNEST(
    CAST(json_parse(per_feature_drift_scores) AS MAP(VARCHAR, DOUBLE))
) AS t(feature_name, drift_score)
WHERE per_feature_drift_scores IS NOT NULL
    AND per_feature_drift_scores != 'null'
    AND per_feature_drift_scores != '{{}}'
"""

# Execute view creation
response = athena.start_query_execution(
    QueryString=create_view_sql,
    QueryExecutionContext={'Database': ATHENA_DATABASE},
    ResultConfiguration={'OutputLocation': ATHENA_OUTPUT_S3}
)

query_id = response['QueryExecutionId']
print(f"Query execution ID: {query_id}")

# Wait for completion
while True:
    result = athena.get_query_execution(QueryExecutionId=query_id)
    status = result['QueryExecution']['Status']['State']
    if status == 'SUCCEEDED':
        print("✓ View created successfully!")
        break
    elif status in ['FAILED', 'CANCELLED']:
        reason = result['QueryExecution']['Status'].get('StateChangeReason', 'Unknown')
        print(f"✗ Failed: {reason}")
        raise Exception(f"View creation failed: {reason}")
    time.sleep(2)

# Test the view
print("\nTesting view with sample query...")
test_query = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT feature_name) as features,
    COUNT(DISTINCT monitoring_run_id) as runs,
    MIN(monitoring_timestamp) as first_run,
    MAX(monitoring_timestamp) as last_run
FROM {ATHENA_DATABASE}.feature_drift_detail
"""

response = athena.start_query_execution(
    QueryString=test_query,
    QueryExecutionContext={'Database': ATHENA_DATABASE},
    ResultConfiguration={'OutputLocation': ATHENA_OUTPUT_S3}
)

query_id = response['QueryExecutionId']
while True:
    result = athena.get_query_execution(QueryExecutionId=query_id)
    status = result['QueryExecution']['Status']['State']
    if status == 'SUCCEEDED':
        results = athena.get_query_results(QueryExecutionId=query_id)
        if len(results['ResultSet']['Rows']) > 1:
            data = results['ResultSet']['Rows'][1]['Data']
            print(f"✓ View test successful!")
            print(f"  Total rows: {data[0].get('VarCharValue', '0')}")
            print(f"  Unique features: {data[1].get('VarCharValue', '0')}")
            print(f"  Monitoring runs: {data[2].get('VarCharValue', '0')}")
            print(f"  First run: {data[3].get('VarCharValue', 'N/A')}")
            print(f"  Last run: {data[4].get('VarCharValue', 'N/A')}")
        break
    elif status in ['FAILED', 'CANCELLED']:
        reason = result['QueryExecution']['Status'].get('StateChangeReason', 'Unknown')
        print(f"✗ Test query failed: {reason}")
        break
    time.sleep(2)

print("\n✓ View is ready for QuickSight dataset!")

Creating feature_drift_detail view...
Query execution ID: 557e37c1-eaba-4b09-ae6f-e126db31d3bc
✓ View created successfully!

Testing view with sample query...
✓ View test successful!
  Total rows: 174
  Unique features: 29
  Monitoring runs: 6
  First run: 2026-03-21 09:51:26.000000
  Last run: 2026-03-24 07:37:08.000000

✓ View is ready for QuickSight dataset!


In [10]:
# Grant Lake Formation permissions on the feature_drift_detail VIEW
# Views require separate permissions from underlying tables

import subprocess
import json

print("Granting Lake Formation permissions on feature_drift_detail view...")

# Grant to IAM_ALLOWED_PRINCIPALS (allows any IAM role with Athena/Glue permissions)
resource_json = json.dumps({
    'Table': {
        'DatabaseName': ATHENA_DATABASE,
        'Name': 'feature_drift_detail'
    }
})

cmd = [
    'aws', 'lakeformation', 'grant-permissions',
    '--principal', '{"DataLakePrincipalIdentifier":"IAM_ALLOWED_PRINCIPALS"}',
    '--resource', resource_json,
    '--permissions', 'SELECT', 'DESCRIBE'
]

try:
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    if result.returncode == 0:
        print("✓ Lake Formation permissions granted on view")
        print("  Principal: IAM_ALLOWED_PRINCIPALS")
        print("  Resource: feature_drift_detail")
        print("  Permissions: SELECT, DESCRIBE")
    else:
        # Check if already granted
        if 'AlreadyExistsException' in result.stderr or 'already exists' in result.stderr.lower():
            print("✓ Permissions already exist (no change needed)")
        else:
            print(f"⚠️  Warning: {result.stderr}")
            print("  This might be OK if permissions were granted previously")
except subprocess.TimeoutExpired:
    print("⚠️  Command timed out (permissions may still be granted)")
except Exception as e:
    print(f"⚠️  Error: {e}")
    print("  You may need to grant permissions manually in Lake Formation console")

print("\n✓ Ready for QuickSight dataset creation!")

Granting Lake Formation permissions on feature_drift_detail view...
✓ Lake Formation permissions granted on view
  Principal: IAM_ALLOWED_PRINCIPALS
  Resource: feature_drift_detail
  Permissions: SELECT, DESCRIBE

✓ Ready for QuickSight dataset creation!


In [11]:
# Feature-level drift dataset (from Athena view)
# Shows individual feature drift scores across monitoring runs

FEATURE_LEVEL_DATASET_ID = 'fraud-governance-feature-level-dataset'
FEATURE_LEVEL_DATASET_NAME = 'Fraud Governance - Feature Level Drift'

feature_level_physical_table = {
    'feature-level-view': {
        'RelationalTable': {
            'DataSourceArn': DATASOURCE_ARN,
            'Catalog': 'AwsDataCatalog',
            'Schema': ATHENA_DATABASE,
            'Name': 'feature_drift_detail',
            'InputColumns': [
                {'Name': 'monitoring_run_id', 'Type': 'STRING'},
                {'Name': 'monitoring_timestamp', 'Type': 'DATETIME'},
                {'Name': 'model_version', 'Type': 'STRING'},
                {'Name': 'endpoint_name', 'Type': 'STRING'},
                {'Name': 'data_drift_detected', 'Type': 'BIT'},
                {'Name': 'drifted_columns_count', 'Type': 'INTEGER'},
                {'Name': 'drifted_columns_share', 'Type': 'DECIMAL'},
                {'Name': 'baseline_roc_auc', 'Type': 'DECIMAL'},
                {'Name': 'current_roc_auc', 'Type': 'DECIMAL'},
                {'Name': 'feature_name', 'Type': 'STRING'},
                {'Name': 'drift_score', 'Type': 'DECIMAL'},
                {'Name': 'drift_severity', 'Type': 'STRING'},
                {'Name': 'drift_detected', 'Type': 'BIT'},
            ],
        }
    }
}

feature_level_logical_table = {
    'feature-level-logical': {
        'Alias': 'Feature Level Drift',
        'Source': {'PhysicalTableId': 'feature-level-view'},
    }
}

feature_level_dset_common = dict(
    AwsAccountId=ACCOUNT_ID,
    DataSetId=FEATURE_LEVEL_DATASET_ID,
    Name=FEATURE_LEVEL_DATASET_NAME,
    PhysicalTableMap=feature_level_physical_table,
    LogicalTableMap=feature_level_logical_table,
    ImportMode='DIRECT_QUERY',
)

try:
    quicksight.describe_data_set(AwsAccountId=ACCOUNT_ID, DataSetId=FEATURE_LEVEL_DATASET_ID)
    print('Updating existing feature-level dataset...')
    resp = quicksight.update_data_set(**feature_level_dset_common)
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new feature-level dataset...')
        resp = quicksight.create_data_set(
            **feature_level_dset_common,
            Permissions=[{'Principal': QS_PRINCIPAL, 'Actions': DSET_ACTIONS}],
        )
    else:
        raise

FEATURE_LEVEL_DATASET_ARN = resp['Arn']
print(f'✓ Feature-level dataset: {FEATURE_LEVEL_DATASET_ARN}')

Updating existing feature-level dataset...
✓ Feature-level dataset: arn:aws:quicksight:us-east-1:<ACCOUNT_ID>:dataset/fraud-governance-feature-level-dataset


## 7. Define Visuals

**Sheet 1 — Inference Monitoring** (6 visuals from `inference_responses`):
1. Prediction Volume Over Time (line chart)
2. Fraud Probability Distribution (histogram)
3. Prediction Accuracy Breakdown (donut)
4. Risk Tier Distribution (bar chart)
5. Inference Latency Trend (line chart)
6. Total Inferences KPI

**Sheet 2 — Drift Trend Analysis** (6 visuals from `monitoring_responses`):
7. Data Drift Share Over Time
8. Drifted Columns Count Trend
9. ROC-AUC Degradation Trend (baseline vs current)
10. Model Performance Metrics Over Time (accuracy, precision, recall, F1)
11. Drift Alerts Timeline
12. Latest Drift Share KPI

In [12]:
# Helper: column identifier shorthand
def col(name):
    return {'DataSetIdentifier': 'inference-ds', 'ColumnName': name}

# V1: Prediction Volume Over Time (line chart)
v1_volume = {
    'LineChartVisual': {
        'VisualId': 'v1-prediction-volume',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Prediction Volume Over Time'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'date-dim', 'Column': col('request_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'count-id', 'Column': col('probability_fraud'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                    'Colors': [],
                }
            }
        }
    }
}

# V2: Fraud Probability Distribution (histogram)
v2_histogram = {
    'HistogramVisual': {
        'VisualId': 'v2-fraud-prob-dist',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Fraud Probability Distribution'}},
        'ChartConfiguration': {
            'FieldWells': {
                'HistogramAggregatedFieldWells': {
                    'Values': [{'NumericalMeasureField': {'FieldId': 'prob-fraud', 'Column': col('probability_fraud')}}],
                }
            },
            'BinOptions': {'BinCount': {'Value': 20}},
        }
    }
}

# V3: Prediction Accuracy (pie/donut chart)
v3_accuracy = {
    'PieChartVisual': {
        'VisualId': 'v3-prediction-accuracy',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Prediction Accuracy Breakdown'}},
        'ChartConfiguration': {
            'FieldWells': {
                'PieChartAggregatedFieldWells': {
                    'Category': [{'CategoricalDimensionField': {'FieldId': 'acc-cat', 'Column': col('prediction_accuracy')}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'acc-count', 'Column': col('probability_fraud'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                }
            },
            'DonutOptions': {'ArcOptions': {'ArcThickness': 'MEDIUM'}},
        }
    }
}

# V4: Risk Tier Distribution (bar chart)
v4_risk = {
    'BarChartVisual': {
        'VisualId': 'v4-risk-tier',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Risk Tier Distribution'}},
        'ChartConfiguration': {
            'FieldWells': {
                'BarChartAggregatedFieldWells': {
                    'Category': [{'CategoricalDimensionField': {'FieldId': 'risk-cat', 'Column': col('risk_tier')}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'risk-count', 'Column': col('probability_fraud'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                    'Colors': [],
                }
            },
            'Orientation': 'VERTICAL',
        }
    }
}

# V5: Inference Latency Trend (line chart) — uses inference_latency_ms
v5_latency = {
    'LineChartVisual': {
        'VisualId': 'v5-latency-trend',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Inference Latency Trend (ms)'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'lat-date', 'Column': col('request_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'avg-latency', 'Column': col('inference_latency_ms'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                    'Colors': [],
                }
            }
        }
    }
}

# V6: KPI — Total Inferences
v6_kpi = {
    'KPIVisual': {
        'VisualId': 'v6-total-inferences',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Total Inferences'}},
        'ChartConfiguration': {
            'FieldWells': {
                'Values': [{'NumericalMeasureField': {'FieldId': 'kpi-count', 'Column': col('probability_fraud'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
            }
        }
    }
}

INFERENCE_VISUALS = [v1_volume, v2_histogram, v3_accuracy, v4_risk, v5_latency, v6_kpi]
print(f'Defined {len(INFERENCE_VISUALS)} inference visuals')

Defined 6 inference visuals


### 7b. Drift Trend Analysis Visuals (Sheet 2)

In [13]:
# Helper: column identifier for drift dataset
def dcol(name):
    return {'DataSetIdentifier': 'drift-ds', 'ColumnName': name}

# V7: Data Drift Share Over Time (line chart)
v7_drift_share = {
    'LineChartVisual': {
        'VisualId': 'v7-drift-share-trend',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Data Drift Share Over Time'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'drift-date', 'Column': dcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'drift-share-val', 'Column': dcol('drifted_columns_share'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                    'Colors': [],
                }
            },
            'PrimaryYAxisDisplayOptions': {'AxisOptions': {'AxisLineVisibility': 'VISIBLE'}},
        }
    }
}

# V8: Drifted Columns Count Trend (bar chart)
v8_drifted_count = {
    'BarChartVisual': {
        'VisualId': 'v8-drifted-count-trend',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Drifted Features Count by Monitoring Run'}},
        'ChartConfiguration': {
            'FieldWells': {
                'BarChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'dc-date', 'Column': dcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'dc-count', 'Column': dcol('drifted_columns_count'), 'AggregationFunction': {'SimpleNumericalAggregation': 'MAX'}}}],
                    'Colors': [],
                }
            },
            'Orientation': 'VERTICAL',
        }
    }
}

# V9: ROC-AUC Degradation Trend (multi-line: baseline vs current)
v9_roc_auc = {
    'LineChartVisual': {
        'VisualId': 'v9-roc-auc-trend',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'ROC-AUC: Baseline vs Current Over Time'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'roc-date', 'Column': dcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [
                        {'NumericalMeasureField': {'FieldId': 'baseline-roc', 'Column': dcol('baseline_roc_auc'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}},
                        {'NumericalMeasureField': {'FieldId': 'current-roc', 'Column': dcol('current_roc_auc'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}},
                    ],
                    'Colors': [],
                }
            },
            'PrimaryYAxisDisplayOptions': {'AxisOptions': {'AxisLineVisibility': 'VISIBLE'}},
        }
    }
}

# V10: Model Performance Metrics Over Time (multi-line: accuracy, precision, recall, F1)
v10_perf_metrics = {
    'LineChartVisual': {
        'VisualId': 'v10-perf-metrics-trend',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Model Performance Metrics Over Time'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'perf-date', 'Column': dcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [
                        {'NumericalMeasureField': {'FieldId': 'perf-accuracy', 'Column': dcol('accuracy'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}},
                        {'NumericalMeasureField': {'FieldId': 'perf-precision', 'Column': dcol('precision'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}},
                        {'NumericalMeasureField': {'FieldId': 'perf-recall', 'Column': dcol('recall'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}},
                        {'NumericalMeasureField': {'FieldId': 'perf-f1', 'Column': dcol('f1_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}},
                    ],
                    'Colors': [],
                }
            },
            'PrimaryYAxisDisplayOptions': {'AxisOptions': {'AxisLineVisibility': 'VISIBLE'}},
        }
    }
}

# V11: Drift Alerts Timeline (bar chart of alert_sent events)
v11_alerts = {
    'BarChartVisual': {
        'VisualId': 'v11-drift-alerts',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Drift Alerts Timeline'}},
        'ChartConfiguration': {
            'FieldWells': {
                'BarChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'alert-date', 'Column': dcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'alert-count', 'Column': dcol('drifted_columns_count'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                    'Colors': [{'CategoricalDimensionField': {'FieldId': 'alert-flag', 'Column': dcol('drift_severity')}}],
                }
            },
            'Orientation': 'VERTICAL',
        }
    }
}

# V12: Latest Drift Share KPI
v12_drift_kpi = {
    'KPIVisual': {
        'VisualId': 'v12-drift-kpi',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Latest Drift Share'}},
        'ChartConfiguration': {
            'FieldWells': {
                'Values': [{'NumericalMeasureField': {'FieldId': 'kpi-drift', 'Column': dcol('drifted_columns_share'), 'AggregationFunction': {'SimpleNumericalAggregation': 'MAX'}}}],
            }
        }
    }
}

DRIFT_VISUALS = [v7_drift_share, v8_drifted_count, v9_roc_auc, v10_perf_metrics, v11_alerts, v12_drift_kpi]
print(f'Defined {len(DRIFT_VISUALS)} drift trend visuals')

Defined 6 drift trend visuals


### 7c. Feature Drift Analysis Visuals (Sheet 3)

Shows drift trends by model version and correlation with inference volume.

In [14]:
# Helper: column identifier for feature drift dataset
def fcol(name):
    return {'DataSetIdentifier': 'feature-drift-ds', 'ColumnName': name}

# V13: Drift by Model Version Over Time (multi-line chart)
v13_drift_by_version = {
    'LineChartVisual': {
        'VisualId': 'v13-drift-by-version',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Drift by Model Version Over Time'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'fdrift-date', 'Column': fcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'fdrift-share', 'Column': fcol('drifted_columns_share'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                    'Colors': [{'CategoricalDimensionField': {'FieldId': 'model-version', 'Column': fcol('model_version')}}],
                }
            },
            'PrimaryYAxisDisplayOptions': {'AxisOptions': {'AxisLineVisibility': 'VISIBLE'}},
        }
    }
}

# V14: Model Version Comparison Table
v14_model_comparison = {
    'TableVisual': {
        'VisualId': 'v14-model-comparison',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Model Version Performance Summary'}},
        'ChartConfiguration': {
            'FieldWells': {
                'TableAggregatedFieldWells': {
                    'GroupBy': [{'CategoricalDimensionField': {'FieldId': 'model-ver', 'Column': fcol('model_version')}}],
                    'Values': [
                        {'NumericalMeasureField': {'FieldId': 'latest-drift', 'Column': fcol('drifted_columns_share'), 'AggregationFunction': {'SimpleNumericalAggregation': 'MAX'}}},
                        {'NumericalMeasureField': {'FieldId': 'avg-roc', 'Column': fcol('current_roc_auc'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}},
                        {'NumericalMeasureField': {'FieldId': 'run-count', 'Column': fcol('drifted_columns_count'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}},
                        {'DateMeasureField': {'FieldId': 'last-check', 'Column': fcol('monitoring_timestamp'), 'AggregationFunction': 'MAX'}},
                    ],
                }
            },
            'SortConfiguration': {
                'RowSort': [{
                    'FieldSort': {
                        'FieldId': 'last-check',
                        'Direction': 'DESC'
                    }
                }]
            },
        }
    }
}

# V15: Inference Volume vs Drift Correlation (combo chart)
v15_volume_drift_correlation = {
    'ComboChartVisual': {
        'VisualId': 'v15-volume-drift-corr',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Inference Volume vs Drift Correlation'}},
        'ChartConfiguration': {
            'FieldWells': {
                'ComboChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'corr-date', 'Column': fcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'BarValues': [{'NumericalMeasureField': {'FieldId': 'inf-count', 'Column': fcol('inference_count'), 'AggregationFunction': {'SimpleNumericalAggregation': 'SUM'}}}],
                    'LineValues': [{'NumericalMeasureField': {'FieldId': 'corr-drift', 'Column': fcol('drifted_columns_share'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                    'Colors': [],
                }
            },
            'BarsArrangement': 'CLUSTERED',
        }
    }
}

# V16: Drift Heatmap (Pivot Table)
v16_drift_heatmap = {
    'PivotTableVisual': {
        'VisualId': 'v16-drift-heatmap',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Drift Intensity Heatmap (by Time & Version)'}},
        'ChartConfiguration': {
            'FieldWells': {
                'PivotTableAggregatedFieldWells': {
                    'Rows': [{'DateDimensionField': {'FieldId': 'hmap-date', 'Column': fcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Columns': [{'CategoricalDimensionField': {'FieldId': 'hmap-version', 'Column': fcol('model_version')}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'hmap-drift', 'Column': fcol('drifted_columns_share'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                }
            },
        }
    }
}

FEATURE_DRIFT_VISUALS = [v13_drift_by_version, v14_model_comparison, v15_volume_drift_correlation, v16_drift_heatmap]
print(f'Defined {len(FEATURE_DRIFT_VISUALS)} feature drift visuals')

Defined 4 feature drift visuals


In [15]:
# Helper: column identifier for feature-level dataset
def flcol(name):
    return {'DataSetIdentifier': 'feature-level-ds', 'ColumnName': name}

# V17: Feature Drift Timeline (line chart - drift scores over time by feature)
v17_feature_timeline = {
    'LineChartVisual': {
        'VisualId': 'v17-feature-timeline',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Feature Drift Score Timeline'}},
        'ChartConfiguration': {
            'FieldWells': {
                'LineChartAggregatedFieldWells': {
                    'Category': [{'DateDimensionField': {'FieldId': 'time', 'Column': flcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'drift', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                    'Colors': [{'CategoricalDimensionField': {'FieldId': 'feature', 'Column': flcol('feature_name')}}],
                }
            },
            'SortConfiguration': {},
            'Type': 'LINE',
            'Legend': {'Visibility': 'VISIBLE', 'Position': 'RIGHT'},
            'PrimaryYAxisDisplayOptions': {'AxisOptions': {'AxisLineVisibility': 'VISIBLE'}},
        }
    }
}

# V18: Top Drifting Features (horizontal bar - avg drift by feature)
v18_top_features = {
    'BarChartVisual': {
        'VisualId': 'v18-top-features',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Top 15 Drifting Features'}},
        'ChartConfiguration': {
            'FieldWells': {
                'BarChartAggregatedFieldWells': {
                    'Category': [{'CategoricalDimensionField': {'FieldId': 'feature', 'Column': flcol('feature_name')}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'drift', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                }
            },
            'SortConfiguration': {
                'CategoryItemsLimit': {'OtherCategories': 'INCLUDE', 'ItemsLimit': 15},
                'CategorySort': [{'FieldSort': {'FieldId': 'drift', 'Direction': 'DESC'}}],
            },
            'Orientation': 'HORIZONTAL',
            'BarsArrangement': 'CLUSTERED',
            'Legend': {'Visibility': 'HIDDEN'},
        }
    }
}

# V19: Feature Drift Detail Table
v19_feature_table = {
    'TableVisual': {
        'VisualId': 'v19-feature-table',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Feature Drift Details'}},
        'ChartConfiguration': {
            'FieldWells': {
                'TableAggregatedFieldWells': {
                    'GroupBy': [
                        {'DateDimensionField': {'FieldId': 'time', 'Column': flcol('monitoring_timestamp'), 'DateGranularity': 'MINUTE'}},
                        {'CategoricalDimensionField': {'FieldId': 'feature', 'Column': flcol('feature_name')}},
                        {'CategoricalDimensionField': {'FieldId': 'severity', 'Column': flcol('drift_severity')}},
                        {'CategoricalDimensionField': {'FieldId': 'model', 'Column': flcol('model_version')}},
                    ],
                    'Values': [
                        {'NumericalMeasureField': {'FieldId': 'drift', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}},
                    ],
                }
            },
            'SortConfiguration': {
                'RowSort': [{'FieldSort': {'FieldId': 'time', 'Direction': 'DESC'}}],
            },
        }
    }
}

# V20: Drift Severity Distribution (stacked bar by severity)
v20_severity_dist = {
    'BarChartVisual': {
        'VisualId': 'v20-severity-dist',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Drift Severity by Feature (Top 15)'}},
        'ChartConfiguration': {
            'FieldWells': {
                'BarChartAggregatedFieldWells': {
                    'Category': [{'CategoricalDimensionField': {'FieldId': 'feature', 'Column': flcol('feature_name')}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'count', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'COUNT'}}}],
                    'Colors': [{'CategoricalDimensionField': {'FieldId': 'severity', 'Column': flcol('drift_severity')}}],
                }
            },
            'SortConfiguration': {
                'CategoryItemsLimit': {'OtherCategories': 'INCLUDE', 'ItemsLimit': 15},
                'CategorySort': [{'FieldSort': {'FieldId': 'count', 'Direction': 'DESC'}}],
            },
            'Orientation': 'HORIZONTAL',
            'BarsArrangement': 'STACKED',
            'Legend': {'Visibility': 'VISIBLE', 'Position': 'RIGHT'},
        }
    }
}

# V21: Highest Drifting Feature KPI
v21_worst_feature = {
    'KPIVisual': {
        'VisualId': 'v21-worst-feature',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Highest Drift Score'}},
        'ChartConfiguration': {
            'FieldWells': {
                'Values': [{'NumericalMeasureField': {'FieldId': 'max-drift', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'MAX'}}}],
            },
            'SortConfiguration': {},
        }
    }
}

# V22: Feature Drift Heatmap (pivot table)
v22_drift_heatmap = {
    'PivotTableVisual': {
        'VisualId': 'v22-drift-heatmap',
        'Title': {'Visibility': 'VISIBLE', 'FormatText': {'PlainText': 'Feature Drift Heatmap (Features × Time)'}},
        'ChartConfiguration': {
            'FieldWells': {
                'PivotTableAggregatedFieldWells': {
                    'Rows': [{'CategoricalDimensionField': {'FieldId': 'feature', 'Column': flcol('feature_name')}}],
                    'Columns': [{'DateDimensionField': {'FieldId': 'date', 'Column': flcol('monitoring_timestamp'), 'DateGranularity': 'DAY'}}],
                    'Values': [{'NumericalMeasureField': {'FieldId': 'drift', 'Column': flcol('drift_score'), 'AggregationFunction': {'SimpleNumericalAggregation': 'AVERAGE'}}}],
                }
            },
            'SortConfiguration': {},
        }
    }
}

FEATURE_LEVEL_VISUALS = [
    v17_feature_timeline,
    v18_top_features,
    v19_feature_table,
    v20_severity_dist,
    v21_worst_feature,
    v22_drift_heatmap
]

print(f'Defined {len(FEATURE_LEVEL_VISUALS)} feature-level visuals')
print('  V17: Feature Drift Score Timeline (line chart with line per feature)')
print('  V18: Top 15 Drifting Features (horizontal bar)')
print('  V19: Feature Drift Details (table)')
print('  V20: Drift Severity by Feature (stacked bar)')
print('  V21: Highest Drift Score (KPI)')
print('  V22: Feature Drift Heatmap (pivot table)')


Defined 6 feature-level visuals
  V17: Feature Drift Score Timeline (line chart with line per feature)
  V18: Top 15 Drifting Features (horizontal bar)
  V19: Feature Drift Details (table)
  V20: Drift Severity by Feature (stacked bar)
  V21: Highest Drift Score (KPI)
  V22: Feature Drift Heatmap (pivot table)


In [16]:
# VERIFICATION: Check visual structure before creating analysis
import json

def verify_visuals():
    """Verify DRIFT_VISUALS and FEATURE_DRIFT_VISUALS have correct structure"""
    issues = []
    
    # Check DRIFT_VISUALS
    for i, visual in enumerate(DRIFT_VISUALS):
        if 'LineChartVisual' in visual:
            config = visual['LineChartVisual']['ChartConfiguration']
            if 'PrimaryYAxisDisplayOptions' in config:
                opts = config['PrimaryYAxisDisplayOptions']
                if 'AxisLineVisibility' in opts:
                    issues.append(f"DRIFT_VISUALS[{i}]: AxisLineVisibility at top level (needs AxisOptions wrapper)")
                elif 'AxisOptions' not in opts:
                    issues.append(f"DRIFT_VISUALS[{i}]: Missing AxisOptions wrapper")
    
    # Check FEATURE_DRIFT_VISUALS
    for i, visual in enumerate(FEATURE_DRIFT_VISUALS):
        if 'LineChartVisual' in visual:
            config = visual['LineChartVisual']['ChartConfiguration']
            if 'PrimaryYAxisDisplayOptions' in config:
                opts = config['PrimaryYAxisDisplayOptions']
                if 'AxisLineVisibility' in opts:
                    issues.append(f"FEATURE_DRIFT_VISUALS[{i}]: AxisLineVisibility at top level")
                elif 'AxisOptions' not in opts:
                    issues.append(f"FEATURE_DRIFT_VISUALS[{i}]: Missing AxisOptions wrapper")
        
        # Check DateMeasureField
        if 'TableVisual' in visual:
            fw = visual['TableVisual']['ChartConfiguration']['FieldWells']['TableAggregatedFieldWells']
            for val in fw.get('Values', []):
                if 'DateMeasureField' in val:
                    agg = val['DateMeasureField']['AggregationFunction']
                    if isinstance(agg, dict):
                        issues.append(f"FEATURE_DRIFT_VISUALS[{i}]: DateMeasureField AggregationFunction is dict (should be string)")
    
    if issues:
        print("❌ VALIDATION FAILED - Variables have old structure:")
        for issue in issues:
            print(f"  - {issue}")
        print("\n⚠️  You MUST re-run cells 20 and 22 to fix this!")
        print("    The notebook file is correct, but your Python variables are outdated.")
        return False
    else:
        print("✅ VALIDATION PASSED")
        print(f"  - DRIFT_VISUALS: {len(DRIFT_VISUALS)} visuals with correct structure")
        print(f"  - FEATURE_DRIFT_VISUALS: {len(FEATURE_DRIFT_VISUALS)} visuals with correct structure")
        print("\n✓ Safe to proceed with analysis creation (next cell)")
        return True

verify_visuals()

✅ VALIDATION PASSED
  - DRIFT_VISUALS: 6 visuals with correct structure
  - FEATURE_DRIFT_VISUALS: 4 visuals with correct structure

✓ Safe to proceed with analysis creation (next cell)


True

In [17]:
import json
print("inference_id in INFERENCE_VISUALS:",
        "'inference_id'" in json.dumps(INFERENCE_VISUALS))
print("probability_fraud in INFERENCE_VISUALS:",
        "'probability_fraud'" in json.dumps(INFERENCE_VISUALS))

inference_id in INFERENCE_VISUALS: False
probability_fraud in INFERENCE_VISUALS: False


## 8. Create Analysis via Definition API

In [18]:
# ⚠️ DIAGNOSTIC: Check if variables have been updated
# Run this BEFORE Cell 25 to verify you've re-run cells 18, 20, 22

import json

print("=" * 80)
print("CHECKING VARIABLE VALUES IN KERNEL MEMORY")
print("=" * 80)

issues = []

# Check INFERENCE_VISUALS
inference_str = json.dumps(INFERENCE_VISUALS)
if "'inference_id'" in inference_str:
    issues.append("❌ INFERENCE_VISUALS still uses 'inference_id' (Cell 18 NOT re-run)")
elif "'probability_fraud'" in inference_str:
    print("✅ INFERENCE_VISUALS: Uses probability_fraud (Cell 18 was re-run)")

# Check DRIFT_VISUALS
drift_str = json.dumps(DRIFT_VISUALS)
if "'monitoring_run_id'" in drift_str:
    issues.append("❌ DRIFT_VISUALS still uses 'monitoring_run_id' (Cell 20 NOT re-run)")
elif "'drifted_columns_count'" in drift_str:
    print("✅ DRIFT_VISUALS: Uses drifted_columns_count (Cell 20 was re-run)")

# Check FEATURE_DRIFT_VISUALS
feature_str = json.dumps(FEATURE_DRIFT_VISUALS)
if "'monitoring_run_id'" in feature_str:
    issues.append("❌ FEATURE_DRIFT_VISUALS still uses 'monitoring_run_id' (Cell 22 NOT re-run)")
elif "'drifted_columns_count'" in feature_str:
    print("✅ FEATURE_DRIFT_VISUALS: Uses drifted_columns_count (Cell 22 was re-run)")

print("\n" + "=" * 80)

if issues:
    print("\n⚠️  VARIABLES NOT UPDATED - Analysis will fail!")
    print("\nProblems found:")
    for issue in issues:
        print(f"  {issue}")
    
    print("\n🔧 TO FIX:")
    print("  1. Go back and run Cell 18 (defines INFERENCE_VISUALS)")
    print("  2. Go back and run Cell 20 (defines DRIFT_VISUALS)")
    print("  3. Go back and run Cell 22 (defines FEATURE_DRIFT_VISUALS)")
    print("  4. Re-run this diagnostic cell - should show all ✅")
    print("  5. Then run Cell 25 (analysis update)")
    
    print("\n⛔ DO NOT run Cell 25 until all checks show ✅")
else:
    print("\n✅ ALL VARIABLES UPDATED CORRECTLY")
    print("\n✓ Safe to run Cell 25 (analysis update)")
    print("  The analysis should complete successfully now.")

print("=" * 80)


CHECKING VARIABLE VALUES IN KERNEL MEMORY


✅ ALL VARIABLES UPDATED CORRECTLY

✓ Safe to run Cell 25 (analysis update)
  The analysis should complete successfully now.


In [19]:
ANALYSIS_ACTIONS = [
    'quicksight:RestoreAnalysis', 'quicksight:UpdateAnalysisPermissions',
    'quicksight:DeleteAnalysis', 'quicksight:DescribeAnalysisPermissions',
    'quicksight:QueryAnalysis', 'quicksight:DescribeAnalysis', 'quicksight:UpdateAnalysis',
]

analysis_definition = {
    'DataSetIdentifierDeclarations': [
        {'Identifier': 'inference-ds', 'DataSetArn': DATASET_ARN},
        {'Identifier': 'drift-ds', 'DataSetArn': DRIFT_DATASET_ARN},
        {'Identifier': 'feature-drift-ds', 'DataSetArn': FEATURE_DRIFT_DATASET_ARN},
        {'Identifier': 'feature-level-ds', 'DataSetArn': FEATURE_LEVEL_DATASET_ARN},
    ],
    'Sheets': [
        {
            'SheetId': 'governance-sheet-1',
            'Name': 'Inference Monitoring',
            'Visuals': INFERENCE_VISUALS,
        },
        {
            'SheetId': 'governance-sheet-2',
            'Name': 'Drift Trend Analysis',
            'Visuals': DRIFT_VISUALS,
        },
        {
            'SheetId': 'governance-sheet-3',
            'Name': 'Feature Drift Analysis',
            'Visuals': FEATURE_DRIFT_VISUALS,
        },
        {
            'SheetId': 'governance-sheet-4',
            'Name': 'Feature Drift Detail',
            'Visuals': FEATURE_LEVEL_VISUALS,
        },
    ],
    'AnalysisDefaults': {
        'DefaultNewSheetConfiguration': {
            'InteractiveLayoutConfiguration': {
                'FreeForm': {'CanvasSizeOptions': {'ScreenCanvasSizeOptions': {'OptimizedViewPortWidth': '1600px'}}}
            }
        }
    },
}

import time

try:
    quicksight.describe_analysis(AwsAccountId=ACCOUNT_ID, AnalysisId=ANALYSIS_ID)
    print('Analysis exists, updating...')
    resp = quicksight.update_analysis(
        AwsAccountId=ACCOUNT_ID, AnalysisId=ANALYSIS_ID,
        Name=QUICKSIGHT_ANALYSIS_NAME,
        Definition=analysis_definition,
    )
    
    # Wait for analysis to reach terminal state
    print('  Waiting for analysis to complete...')
    max_attempts = 30
    for attempt in range(max_attempts):
        analysis_resp = quicksight.describe_analysis(AwsAccountId=ACCOUNT_ID, AnalysisId=ANALYSIS_ID)
        status = analysis_resp['Analysis']['Status']
        
        if status == 'CREATION_SUCCESSFUL':
            print(f'  ✓ Analysis update successful')
            break
        elif status in ['CREATION_FAILED', 'UPDATE_FAILED', 'DELETED']:
            print(f'  ✗ Analysis update failed with status: {status}')
            
            # Get error details
            if 'Errors' in analysis_resp['Analysis']:
                errors = analysis_resp['Analysis']['Errors']
                print(f'\n  Errors ({len(errors)} total):')
                for i, err in enumerate(errors[:5], 1):  # Show first 5 errors
                    err_type = err.get('Type', 'Unknown')
                    err_msg = err.get('Message', 'No message')
                    print(f'    {i}. [{err_type}] {err_msg}')
                
                if len(errors) > 5:
                    print(f'    ... and {len(errors) - 5} more errors')
            
            raise Exception(f"Analysis in {status} state - see errors above")
        
        time.sleep(2)
    else:
        print(f'  ⚠ Timeout waiting for analysis (status: {status})')
        
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new analysis...')
        resp = quicksight.create_analysis(
            AwsAccountId=ACCOUNT_ID, AnalysisId=ANALYSIS_ID,
            Name=QUICKSIGHT_ANALYSIS_NAME,
            Definition=analysis_definition,
            Permissions=[{'Principal': QS_PRINCIPAL, 'Actions': ANALYSIS_ACTIONS}],
        )
        print('  ✓ Analysis created')
    else:
        raise

ANALYSIS_ARN = resp['Arn']
print(f'\n✓ Analysis: {ANALYSIS_ARN}')
print(f'  Open: https://{REGION}.quicksight.aws.amazon.com/sn/analyses/{ANALYSIS_ID}')

Analysis exists, updating...
  Waiting for analysis to complete...
  ⚠ Timeout waiting for analysis (status: UPDATE_SUCCESSFUL)

✓ Analysis: arn:aws:quicksight:us-east-1:<ACCOUNT_ID>:analysis/fraud-governance-analysis
  Open: https://us-east-1.quicksight.aws.amazon.com/sn/analyses/fraud-governance-analysis


## 9. Publish Dashboard via Definition API

In [ ]:
DASHBOARD_ACTIONS = [
    'quicksight:DescribeDashboard', 'quicksight:ListDashboardVersions',
    'quicksight:UpdateDashboardPermissions', 'quicksight:QueryDashboard',
    'quicksight:UpdateDashboard', 'quicksight:DeleteDashboard',
    'quicksight:DescribeDashboardPermissions', 'quicksight:UpdateDashboardPublishedVersion',
]

dashboard_definition = {
    'DataSetIdentifierDeclarations': [
        {'Identifier': 'inference-ds', 'DataSetArn': DATASET_ARN},
        {'Identifier': 'drift-ds', 'DataSetArn': DRIFT_DATASET_ARN},
        {'Identifier': 'feature-drift-ds', 'DataSetArn': FEATURE_DRIFT_DATASET_ARN},
        {'Identifier': 'feature-level-ds', 'DataSetArn': FEATURE_LEVEL_DATASET_ARN},
    ],
    'Sheets': [
        {
            'SheetId': 'governance-sheet-1',
            'Name': 'Inference Monitoring',
            'Visuals': INFERENCE_VISUALS,
        },
        {
            'SheetId': 'governance-sheet-2',
            'Name': 'Drift Trend Analysis',
            'Visuals': DRIFT_VISUALS,
        },
        {
            'SheetId': 'governance-sheet-3',
            'Name': 'Feature Drift Analysis',
            'Visuals': FEATURE_DRIFT_VISUALS,
        },
        {
            'SheetId': 'governance-sheet-4',
            'Name': 'Feature Drift Detail',
            'Visuals': FEATURE_LEVEL_VISUALS,
        },
    ],
}

publish_options = {
    'AdHocFilteringOption': {'AvailabilityStatus': 'ENABLED'},
    'ExportToCSVOption': {'AvailabilityStatus': 'ENABLED'},
    'SheetControlsOption': {'VisibilityState': 'EXPANDED'},
}

import time

try:
    quicksight.describe_dashboard(AwsAccountId=ACCOUNT_ID, DashboardId=DASHBOARD_ID)
    print('Dashboard exists, updating...')
    resp = quicksight.update_dashboard(
        AwsAccountId=ACCOUNT_ID, DashboardId=DASHBOARD_ID,
        Name=QUICKSIGHT_DASHBOARD_NAME,
        Definition=dashboard_definition,
        DashboardPublishOptions=publish_options,
    )
    version = resp['VersionArn'].split('/')[-1]
    
    # Wait for dashboard to reach terminal state
    print(f'  Waiting for dashboard version {version} to complete...')
    max_attempts = 30
    for attempt in range(max_attempts):
        dash_resp = quicksight.describe_dashboard(
            AwsAccountId=ACCOUNT_ID, 
            DashboardId=DASHBOARD_ID,
            VersionNumber=int(version)
        )
        status = dash_resp['Dashboard']['Version']['Status']
        
        if status == 'CREATION_SUCCESSFUL':
            print(f'  ✓ Dashboard update successful')
            break
        elif status in ['CREATION_FAILED', 'UPDATE_FAILED', 'DELETED']:
            error_info = dash_resp['Dashboard']['Version'].get('Errors', [])
            print(f'  ✗ Dashboard update failed with status: {status}')
            if error_info:
                print(f'  Errors:')
                for err in error_info[:3]:  # Show first 3 errors
                    print(f"    - {err.get('Type', 'Unknown')}: {err.get('Message', 'No message')}")
            raise Exception(f"Dashboard in {status} state, cannot publish")
        
        time.sleep(2)
    else:
        print(f'  ⚠ Timeout waiting for dashboard (status: {status})')
    
    # Publish if successful
    if status == 'CREATION_SUCCESSFUL':
        quicksight.update_dashboard_published_version(
            AwsAccountId=ACCOUNT_ID, DashboardId=DASHBOARD_ID, VersionNumber=int(version)
        )
        print(f'  ✓ Published version {version}')
        
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('Creating new dashboard...')
        resp = quicksight.create_dashboard(
            AwsAccountId=ACCOUNT_ID, DashboardId=DASHBOARD_ID,
            Name=QUICKSIGHT_DASHBOARD_NAME,
            Definition=dashboard_definition,
            DashboardPublishOptions=publish_options,
            Permissions=[{'Principal': QS_PRINCIPAL, 'Actions': DASHBOARD_ACTIONS}],
        )
        version = resp['VersionArn'].split('/')[-1]
        print(f'  ✓ Created dashboard version {version}')
    else:
        raise

dashboard_url = f'https://{REGION}.quicksight.aws.amazon.com/sn/dashboards/{DASHBOARD_ID}'
print(f'\n✓ Dashboard: {dashboard_url}')

Dashboard exists, updating...
  Waiting for dashboard version 2 to complete...
  ✓ Dashboard update successful
  ✓ Published version 2

✓ Dashboard: https://us-east-1.quicksight.aws.amazon.com/sn/dashboards/fraud-governance-dashboard


## 10. Generate Embed URL (Optional)

In [19]:
try:
    resp = quicksight.get_dashboard_embed_url(
        AwsAccountId=ACCOUNT_ID, DashboardId=DASHBOARD_ID,
        IdentityType='QUICKSIGHT', SessionLifetimeInMinutes=600,
        UndoRedoDisabled=False, ResetDisabled=False,
    )
    print(f'Embed URL (valid 10 hours):\n  {resp["EmbedUrl"]}')
    print(f'\nHTML:\n  <iframe src="{resp["EmbedUrl"]}" width="100%" height="800px"></iframe>')
except ClientError as e:
    print(f'Could not generate embed URL: {e.response["Error"]["Message"]}')
    print('Ensure embedding is enabled in QuickSight admin settings.')

Could not generate embed URL: Invalid QuickSight user ARN: null
Ensure embedding is enabled in QuickSight admin settings.


## 11. Cleanup (Optional)

Uncomment and run to delete all QuickSight resources created by this notebook.

In [20]:
# CONFIRM_DELETE = False  # Set True to delete
#
# if CONFIRM_DELETE:
#     for res_type, res_id, id_key in [
#         ('dashboard', DASHBOARD_ID, 'DashboardId'),
#         ('analysis', ANALYSIS_ID, 'AnalysisId'),
#         ('data_set', DATASET_ID, 'DataSetId'),
#         ('data_set', DRIFT_DATASET_ID, 'DataSetId'),
#        ('data_set', QUICKSIGHT_FEATURE_DRIFT_DATASET_ID, 'DataSetId'),
#         ('data_source', DATASOURCE_ID, 'DataSourceId'),
#     ]:
#         try:
#             getattr(quicksight, f'delete_{res_type}')(AwsAccountId=ACCOUNT_ID, **{id_key: res_id})
#             print(f'  \u2713 Deleted {res_type}: {res_id}')
#         except ClientError as e:
#             if e.response['Error']['Code'] != 'ResourceNotFoundException':
#                 print(f'  \u26a0 {res_type}: {e}')
#     print('\u2713 Cleanup complete')

print('Cleanup cell \u2014 uncomment and set CONFIRM_DELETE = True to run')

Cleanup cell — uncomment and set CONFIRM_DELETE = True to run
